In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (training continues, Hub push will skip):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")

In [ ]:
!cd /kaggle/working/arc && bash scripts/preflight_check.sh /kaggle/working/jetmoe-8b configs/phase1/tier_a.yaml

In [ ]:
!cd /kaggle/working/arc && python scripts/train_stage_a.py --config configs/phase1/tier_a.yaml --variant model_adaptive --dry-run --run-id smoke

In [ ]:
!cd /kaggle/working/arc && python scripts/train_stage_a.py --config configs/phase1/tier_a.yaml --variant all --tier a --run-id phase1-tier-a-r2

In [ ]:
import json, glob
for f in sorted(glob.glob("/kaggle/working/checkpoints/phase1-tier-a/phase1-tier-a-r2/*/best.json")):
    print(f, "->", json.load(open(f)))
!cd /kaggle/working/arc && python scripts/check_g2_lite.py --run-dir /kaggle/working/checkpoints/phase1-tier-a/phase1-tier-a-r2

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN absent: skipping Hub push, keeping Kaggle outputs")
else:
    import subprocess
    cmd = "cd /kaggle/working/arc && for v in model_adaptive block_adaptive layer_adaptive; do python scripts/push_best_to_hub.py --repo-id Nyvo/arc-jetmoe-recurrence-phase1 --run-dir /kaggle/working/checkpoints/phase1-tier-a/phase1-tier-a-r2/$v --tag phase1-tierA-$v-best; done"
    print("pushing best adapters to Hub...")
    r = subprocess.run(cmd, shell=True)
    print("push exit:", r.returncode)


In [ ]:
!ls -R /kaggle/working/checkpoints/phase1-tier-a/phase1-tier-a-r2 2>/dev/null | head -40